# 年周視差から恒星までの距離を求める

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/coxless/astro-sandbox/blob/main/distance/parallax.ipynb)

地球が太陽のまわりを公転すると、近くの恒星は遠くの背景に対してわずかに位置がずれて見えます。
このずれの半分の角度が **年周視差** $p$ です。1 au を底辺とした三角形から

$$
d\,[\mathrm{pc}] = \frac{1}{p\,[\mathrm{arcsec}]} = \frac{1000}{p\,[\mathrm{mas}]}
$$

で距離が求まります(1 pc は「視差がちょうど 1 秒角になる距離」の定義)。

このノートブックでやること:

1. 有名な近傍星の視差を Gaia DR3 から取得し、`1/p` で距離を出す
2. 視差の誤差が大きいと `1/p` がどう偏るかを、シミュレーションで確かめる
3. ランダムに選んだ Gaia の星で、`1/p` と Bailer-Jones et al. (2021) のベイズ推定距離を比べる

## 使うデータ(すべてオープン)

- **Gaia DR3** (ESA): `gaiadr3.gaia_source` — 視差・等級など
- **Bailer-Jones et al. (2021)**: `external.gaiaedr3_distance` — Gaia EDR3 の視差から推定した距離(DR3 と source_id 共通)
- **SIMBAD** (CDS): 星の名前から Gaia DR3 の source_id を引くのに使用

## 0. 準備

In [ ]:
# Colab には astroquery が入っていないのでインストールする
!pip install -q astroquery

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

from astroquery.gaia import Gaia
from astroquery.simbad import Simbad

Gaia.ROW_LIMIT = -1  # 行数制限なし(クエリ側で件数を絞る)

## 1. 有名な近傍星の距離

星の名前を SIMBAD で Gaia DR3 の source_id に変換し、Gaia アーカイブから視差を取ってきます。

> 注: シリウスやベガのような非常に明るい星(G ≲ 3 等)は Gaia では正しく測れず、DR3 に含まれていないことがあります。

In [ ]:
STAR_NAMES = [
    "Proxima Centauri",
    "Barnard's Star",
    "Wolf 359",
    "Lalande 21185",
    "61 Cyg A",
    "Ross 128",
]


def gaia_dr3_id(name):
    # SIMBAD の別名一覧から "Gaia DR3 xxxx" を探す。見つからなければ None
    ids = Simbad.query_objectids(name)
    if ids is None:
        return None
    col = ids.colnames[0]  # astroquery のバージョンで 'ID' / 'id' が変わる
    for s in ids[col]:
        s = str(s)
        if s.startswith("Gaia DR3 "):
            return int(s.split()[-1])
    return None


name_to_id = {}
for name in STAR_NAMES:
    sid = gaia_dr3_id(name)
    if sid is None:
        print(f"{name}: Gaia DR3 に見つからず")
    else:
        name_to_id[name] = sid
name_to_id

In [ ]:
id_list = ", ".join(str(i) for i in name_to_id.values())
query = f'''
SELECT source_id, ra, dec, parallax, parallax_error, parallax_over_error, phot_g_mean_mag, ruwe
FROM gaiadr3.gaia_source
WHERE source_id IN ({id_list})
'''
famous = Gaia.launch_job(query).get_results().to_pandas()
famous.columns = famous.columns.str.lower()

id_to_name = {v: k for k, v in name_to_id.items()}
famous.insert(0, "name", famous["source_id"].map(id_to_name))
famous

視差 [mas] から距離 [pc] を計算します。誤差は、$p$ の誤差 $\sigma_p$ が小さいときの近似

$$
\sigma_d \approx \frac{1000\,\sigma_p}{p^2}
$$

で見積もります(誤差伝播の1次近似)。1 pc ≈ 3.2616 光年です。

In [ ]:
PC_TO_LY = 3.26156

famous["dist_pc"] = 1000.0 / famous["parallax"]
famous["dist_err_pc"] = 1000.0 * famous["parallax_error"] / famous["parallax"] ** 2
famous["dist_ly"] = famous["dist_pc"] * PC_TO_LY

famous.sort_values("dist_pc")[
    ["name", "parallax", "parallax_error", "dist_pc", "dist_err_pc", "dist_ly"]
].round(4)

プロキシマ・ケンタウリは約 1.30 pc(約 4.25 光年)になるはずです。
これらの星は `parallax_over_error`($p/\sigma_p$)が数千以上あり、`1/p` で十分正確に距離が求まります。

## 2. 視差の誤差が大きいと何が起きるか(シミュレーション)

遠い星ほど $p$ は小さくなりますが、測定誤差 $\sigma_p$ はあまり変わりません。
すると $p/\sigma_p$ が小さくなり、`1/p` には次の問題が出てきます。

- 逆数をとる変換が非線形なので、誤差分布がゆがんで距離が偏る
- 測定値の $p$ が 0 やマイナスになることがあり、`1/p` が意味をなさない

真の距離を決めて、Gaia 程度の誤差で視差を「測定」するシミュレーションで確かめます。

In [ ]:
rng = np.random.default_rng(42)

TRUE_DIST_PC = 1000.0           # 真の距離
true_plx = 1000.0 / TRUE_DIST_PC  # = 1 mas
N = 100_000

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, sigma in zip(axes, [0.05, 0.2, 0.5]):  # 視差誤差 [mas]
    plx_obs = rng.normal(true_plx, sigma, N)
    pos = plx_obs > 0
    d_inv = 1000.0 / plx_obs[pos]

    ax.hist(d_inv, bins=np.linspace(0, 4000, 200), color="tab:blue", alpha=0.8)
    ax.axvline(TRUE_DIST_PC, color="k", ls="--", label="true distance")
    ax.axvline(np.median(d_inv), color="tab:red", label="median of 1/p")
    ax.set_title(f"p/σ = {true_plx / sigma:.0f}   (p ≤ 0: {np.mean(~pos):.1%})")
    ax.set_xlabel("distance from 1/p [pc]")
    ax.legend()
axes[0].set_ylabel("count")
plt.tight_layout()
plt.show()

- $p/\sigma_p = 20$ では分布は細く、ほぼ真の距離に集まります。
- $p/\sigma_p = 5$ では遠い側に長い裾が伸びます。
- $p/\sigma_p = 2$ では裾がとても長く、2% ほどの星は視差がマイナスになって距離が出ません。

目安として **$p/\sigma_p \gtrsim 10$**(視差の相対誤差 10% 以下)なら `1/p` で十分、と言われることが多いです。
それより悪い場合は、「星は銀河の中にこういう分布で存在するはず」という事前分布を入れたベイズ推定で距離を出します。
これを Gaia の全天の星に対して行ったのが Bailer-Jones et al. (2021) のカタログです。

## 3. 実データで比較: `1/p` と Bailer-Jones の距離

Gaia DR3 から星をランダムに選び(`random_index` を使う)、Bailer-Jones の幾何学的距離 `r_med_geo` と結合して取得します。
件数を増やすと時間がかかるので、まずは 2 万件程度で試します。

In [ ]:
N_SAMPLE = 20_000

query = f'''
SELECT g.source_id, g.parallax, g.parallax_error, g.parallax_over_error,
       g.phot_g_mean_mag, g.ruwe,
       d.r_med_geo, d.r_lo_geo, d.r_hi_geo
FROM gaiadr3.gaia_source AS g
JOIN external.gaiaedr3_distance AS d USING (source_id)
WHERE g.random_index < {N_SAMPLE}
  AND g.parallax IS NOT NULL
'''
job = Gaia.launch_job_async(query)
sample = job.get_results().to_pandas()
sample.columns = sample.columns.str.lower()
print(len(sample), "件")
sample.head()

In [ ]:
neg_frac = np.mean(sample["parallax"] <= 0)
print(f"視差が 0 以下の星: {neg_frac:.1%}")
print(f"p/σ ≥ 10 の星:     {np.mean(sample['parallax_over_error'] >= 10):.1%}")

plt.figure(figsize=(6, 4))
plt.hist(sample["parallax_over_error"].clip(-10, 100), bins=110)
plt.axvline(10, color="tab:red", ls="--", label="p/σ = 10")
plt.xlabel("parallax_over_error (p/σ)")
plt.ylabel("count")
plt.yscale("log")
plt.legend()
plt.show()

ランダムに選ぶと、ほとんどは遠くて暗い星なので、視差の精度がよい星は少数派です。
次に、視差が正の星について `1/p` と `r_med_geo` を並べます。

In [ ]:
pos = sample[sample["parallax"] > 0].copy()
pos["d_inv"] = 1000.0 / pos["parallax"]
pos["ratio"] = pos["d_inv"] / pos["r_med_geo"]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
sc = ax.scatter(pos["r_med_geo"], pos["d_inv"], c=pos["parallax_over_error"].clip(0.1, 1000),
                s=2, cmap="viridis", norm=LogNorm())
lim = [10, 1e6]
ax.plot(lim, lim, "k--", lw=1, label="1/p = r_med_geo")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlim(lim)
ax.set_ylim(lim)
ax.set_xlabel("Bailer-Jones r_med_geo [pc]")
ax.set_ylabel("1/p [pc]")
ax.legend()
fig.colorbar(sc, ax=ax, label="p/σ")

ax = axes[1]
ax.scatter(pos["parallax_over_error"], pos["ratio"], s=2, alpha=0.3)
ax.axhline(1, color="k", ls="--", lw=1)
ax.axvline(10, color="tab:red", ls="--", lw=1, label="p/σ = 10")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlim(0.1, 1000)
ax.set_ylim(0.05, 50)
ax.set_xlabel("p/σ")
ax.set_ylabel("(1/p) / r_med_geo")
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# p/σ の区間ごとに、1/p と r_med_geo の比(中央値)とばらつきを見る
bins = [0, 1, 2, 5, 10, 20, 50, np.inf]
pos["snr_bin"] = pd.cut(pos["parallax_over_error"], bins)
pos.groupby("snr_bin", observed=True)["ratio"].describe(percentiles=[0.16, 0.5, 0.84])[
    ["count", "16%", "50%", "84%"]
].round(3)

## まとめ

- $p/\sigma_p$ が大きい(10 以上)星では、`1/p` と Bailer-Jones の距離はほぼ一致し、比は 1 付近に集まります。
- $p/\sigma_p$ が小さくなるほど比がばらつき、偏りも出ます。Bailer-Jones の距離は事前分布に引っぱられて、もっともらしい値に落ち着きます。
- 視差だけで「信頼して」距離が言えるのは、Gaia でもおおむね数 kpc までです。それより遠くは、別の方法(標準光源など)が必要になります。

## 発展メモ

- Gaia DR3 の視差には、全体で約 −0.017 mas のゼロ点オフセットがあります(Lindegren et al. 2021)。精密にやるときは補正します。
- `ruwe` が 1.4 を大きく超える星は、連星などでアストロメトリの当てはめが悪い可能性があります。

## 参考・クレジット

- Gaia Collaboration, Vallenari et al. (2023), *Gaia Data Release 3: Summary of the content and survey properties*, A&A 674, A1
- Bailer-Jones et al. (2021), *Estimating distances from parallaxes. V. Geometric and photogeometric distances to 1.47 billion stars in Gaia EDR3*, AJ 161, 147
- Lindegren et al. (2021), *Gaia Early Data Release 3: Parallax bias versus magnitude, colour, and position*, A&A 649, A4
- Wenger et al. (2000), *The SIMBAD astronomical database*, A&AS 143, 9

This work has made use of data from the European Space Agency (ESA) mission *Gaia* (https://www.cosmos.esa.int/gaia), processed by the Gaia Data Processing and Analysis Consortium (DPAC, https://www.cosmos.esa.int/web/gaia/dpac/consortium). This research has made use of the SIMBAD database, operated at CDS, Strasbourg, France.